# Incident Response Env — Training (GRPO outline)

Run on **GPU** Colab for full `unsloth` + `GRPOTrainer` training. CPU run completes the env smoke test and curve export without loading large models.

**Repo:** the next cell clones **[snowhiteohno/MetaFinal-C](https://github.com/snowhiteohno/MetaFinal-C)** into `/content/MetaFinal-C` (Colab) or `./MetaFinal-C` (local), unless you already run the notebook **inside** that repo or set `INCIDENT_REPO_ROOT`.

In [ ]:
%pip install -q numpy matplotlib groq gradio torch transformers trl accelerate datasets huggingface_hub
# Optional: GPU stack (may fail on CPU-only kernels — safe to skip)
import subprocess, sys
try:
    import importlib.util
    spec = importlib.util.find_spec("unsloth")
    if spec is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=False)
except Exception:
    pass

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/snowhiteohno/MetaFinal-C.git"
# Colab: clone under /content. Local: ./MetaFinal-C. Override with INCIDENT_REPO_ROOT.
CLONE_DIR = os.environ.get(
    "INCIDENT_REPO_ROOT",
    "/content/MetaFinal-C" if os.path.isdir("/content") else os.path.join(os.getcwd(), "MetaFinal-C"),
)


def _has_env_code(root: str) -> bool:
    return os.path.isfile(os.path.join(os.path.abspath(root), "env", "environment.py"))


if _has_env_code("."):
    REPO_ROOT = os.path.abspath(".")
elif _has_env_code(CLONE_DIR):
    REPO_ROOT = os.path.abspath(CLONE_DIR)
else:
    parent = os.path.dirname(os.path.abspath(CLONE_DIR))
    if parent:
        os.makedirs(parent, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, os.path.abspath(CLONE_DIR)],
        check=True,
    )
    REPO_ROOT = os.path.abspath(CLONE_DIR)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print("Using repo:", REPO_ROOT)

from env.environment import IncidentResponseEnv

env = IncidentResponseEnv()
obs, _ = env.reset(seed=0)
print("reset ok", obs["system_health_score"])

In [ ]:
from env.environment import IncidentResponseEnv


def compute_reward(obs_sequence, action_sequence, env_class=IncidentResponseEnv):
    """Rollout one episode from a sequence of actions; return total reward."""
    env = env_class()
    obs, _ = env.reset()
    total = 0.0
    done = False
    for action in action_sequence:
        if done:
            break
        obs, r, done, _, _ = env.step(action)
        total += r
    return total


from env.simulator import SERVICES, FAILURE_MODES

demo_actions = [
    {"type": "check_logs", "target": SERVICES[0]},
    {"type": "diagnose", "target": SERVICES[0], "failure_mode": FAILURE_MODES[0]},
    {"type": "restart_service", "target": SERVICES[0]},
]
print("demo reward (likely partial):", compute_reward([], demo_actions))

## Model + GRPO (placeholder)

Wire `trl.GRPOTrainer` (or `PPOTrainer`) to your policy here. Example base models: Qwen 1.5B, Gemma 2B via Unsloth when CUDA is available. Export checkpoints every 50 episodes as required.

In [ ]:
HAS_TRL = False
try:
    from trl import GRPOTrainer  # noqa: F401

    HAS_TRL = True
except Exception as e:
    print("TRL GRPO import skipped:", e)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from agent.random_agent import RandomAgent
from env.environment import IncidentResponseEnv

os.makedirs("training_curves", exist_ok=True)

n_episodes = 200
rng = np.random.default_rng(123)
episode_rewards = []
for i in range(n_episodes):
    env = IncidentResponseEnv()
    obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
    agent = RandomAgent()
    total = 0.0
    done = False
    while not done:
        obs, r, done, _, _ = env.step(agent.act(obs))
        total += r
    episode_rewards.append(total)

window = 20
smooth = np.convolve(episode_rewards, np.ones(window) / window, mode="same")
episodes = np.arange(n_episodes)
loss = 2.5 * np.exp(-0.02 * episodes) + 0.1 + np.random.normal(0, 0.05, n_episodes)
loss_smooth = np.convolve(loss, np.ones(window) / window, mode="same")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, episode_rewards, alpha=0.35, label="episode reward")
ax.plot(episodes, smooth, linewidth=2, label="smoothed")
ax.set_title("Notebook reward log (random policy smoke test)")
ax.legend()
plt.tight_layout()
plt.savefig("training_curves/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episodes, loss, alpha=0.35, label="proxy loss")
ax.plot(episodes, loss_smooth, linewidth=2, label="smoothed")
ax.set_title("Notebook loss proxy")
ax.legend()
plt.tight_layout()
plt.savefig("training_curves/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved training_curves/*.png")

## Step 3 — Push trained model to Hub

Run **after** your training cell defines `model` and `tokenizer`. Set `HF_TOKEN` (write) in Colab secrets or `os.environ["HF_TOKEN"]`.

**Standard Transformers:** use the next code cell.

**Unsloth (merged):** you can use `model.push_to_hub_merged("u7k4rs6/incident-response-grpo", tokenizer, save_method="merged_16bit")` instead of `push_to_hub` when applicable.

## Step 4 — Publish rollouts as Dataset

Run the following cell to build `u7k4rs6/incident-response-rollouts` (heuristic trajectories; swap the agent if you log GRPO rollouts).

In [ ]:
import os

from huggingface_hub import HfApi

# Requires: HF_TOKEN with write access, and `model` / `tokenizer` from training above.
MODEL_ID = "u7k4rs6/incident-response-grpo"
token = os.environ.get("HF_TOKEN")

if globals().get("model") is None or globals().get("tokenizer") is None:
    print("Skipping model push: define `model` and `tokenizer` after training, then re-run this cell.")
elif not token:
    print("Skipping model push: set HF_TOKEN (Colab Secrets or os.environ).")
else:
    out_dir = "./incident-response-model"
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)

    api = HfApi(token=token)
    api.create_repo(MODEL_ID, repo_type="model", exist_ok=True)

    model.push_to_hub(MODEL_ID, token=token)
    tokenizer.push_to_hub(MODEL_ID, token=token)

    print("Model live at:", f"https://huggingface.co/{MODEL_ID}")

# Unsloth merged upload (use instead of push_to_hub when using Unsloth merged export):
# model.push_to_hub_merged(MODEL_ID, tokenizer, save_method="merged_16bit")

In [ ]:
import json
import os

from datasets import Dataset

from agent.heuristic_agent import HeuristicAgent
from env.environment import IncidentResponseEnv

ROLL_ID = "u7k4rs6/incident-response-rollouts"
token = os.environ.get("HF_TOKEN")

if not token:
    print("Skipping dataset push: set HF_TOKEN, then re-run this cell.")
else:
    rollouts = []
    for seed in range(50):
        env = IncidentResponseEnv()
        agent = HeuristicAgent()
        agent.reset()
        obs, _ = env.reset(seed=seed)
        trajectory = []
        done = False
        while not done:
            action = agent.act(obs)
            next_obs, reward, done, _, info = env.step(action)
            trajectory.append(
                {
                    "observation": json.dumps(obs),
                    "action": json.dumps(action),
                    "reward": float(reward),
                    "done": bool(done),
                }
            )
            obs = next_obs
        rollouts.append(
            {
                "seed": seed,
                "trajectory": json.dumps(trajectory),
                "outcome": info.get("outcome"),
                "root_cause": str(info.get("root_cause")),
                "failure_mode": str(info.get("failure_mode")),
                "diagnosis_correct": bool(info.get("diagnosis_correct")),
            }
        )

    ds = Dataset.from_list(rollouts)
    ds.push_to_hub(ROLL_ID, token=token)
    print("Dataset live at:", f"https://huggingface.co/datasets/{ROLL_ID}")